In [1]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

2026-06-05 02:09:19.020748: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-05 02:09:19.397476: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-05 02:09:19.535532: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-05 02:09:19.574121: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-05 02:09:19.815679: I tensorflow/core/platform/cpu_feature_guar

In [2]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [3]:
hidden_layers=[1024, 1024]
epochs=1000
act_func=tf.nn.relu
input_dropout=0.2
hidden_dropout=0.5
learning_rate=0.0001
norm='norm'

seeds = [42, 123, 456, 789, 1011]
results = {'mse': [], 'rmse': [], 'pearson': [], 'mae': []}

In [4]:
train_features, val_features, _, test_features, train_targets, val_targets, _, test_targets = load(norm=norm)

for seed in seeds:
    tf.random.set_seed(seed)
    np.random.seed(seed)
    
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(
                units,
                input_shape=(train_features.shape[1],),
                activation=act_func,
                kernel_initializer='he_normal'))
            if input_dropout > 0:
                model.add(Dropout(float(input_dropout)))
        else:
            model.add(Dense(
                units,
                activation=act_func,
                kernel_initializer='he_normal'))
            if hidden_dropout > 0:
                model.add(Dropout(float(hidden_dropout)))
    model.add(Dense(1, activation='linear', kernel_initializer='he_normal'))

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='mean_squared_error', optimizer=optimizer)

    checkpoint_dir = Path("checkpoints")
    checkpoint_dir.mkdir(exist_ok=True)
    checkpoint_path = Path(f"checkpoints/final_model_seed{seed}.h5")

    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=200,
            restore_best_weights=True,
            verbose=0
        ),
        ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor='val_loss',
            save_best_only=True,
            save_weights_only=False,
            verbose=0
        )
    ]
    history = model.fit(        
        train_features, train_targets,
        validation_data=(val_features, val_targets),
        epochs=epochs,
        batch_size=64,
        callbacks=callbacks,
        shuffle=True,
        verbose=0
    )
    print(f"Seed {seed} - Best Val Loss: {min(history.history['val_loss']):.4f}")
    if checkpoint_path.exists():
        model = tf.keras.models.load_model(str(checkpoint_path))
    test_predictions = model.predict(test_features).flatten()    
    test_targets_flat = test_targets.flatten()
    mse = mean_squared_error(test_targets_flat, test_predictions)
    results['mse'].append(mse)
    results['rmse'].append(np.sqrt(mse))
    results['pearson'].append(pearsonr(test_targets_flat, test_predictions)[0])
    results['mae'].append(mean_absolute_error(test_targets_flat, test_predictions))
    print(f"Seed {seed} done")

I0000 00:00:1780625372.689518     224 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1780625373.470398     224 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1780625373.470430     224 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1780625373.473355     224 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1780625373.473375     224 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

Seed 42 - Best Val Loss: 278.3748
143/143 [==============================] - 0s 1ms/step
Seed 42 done


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

Seed 123 - Best Val Loss: 281.2088
143/143 [==============================] - 0s 1ms/step
Seed 123 done


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

Seed 456 - Best Val Loss: 280.0912
143/143 [==============================] - 0s 1ms/step
Seed 456 done


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

Seed 789 - Best Val Loss: 280.0671
143/143 [==============================] - 0s 1ms/step
Seed 789 done


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

Seed 1011 - Best Val Loss: 277.1517
143/143 [==============================] - 0s 1ms/step
Seed 1011 done


In [5]:
print("FINAL RESULTS")
for metric, values in results.items():
    print(f"{metric.upper()}: {np.mean(values):.4f} ± {np.std(values):.4f}")

FINAL RESULTS
MSE: 274.0967 ± 5.5339
RMSE: 16.5550 ± 0.1672
PEARSON: 0.6057 ± 0.0115
MAE: 10.8589 ± 0.0979
